In [1]:
from pathlib import Path
from contextlib import contextmanager
from time import perf_counter
import gc
import os
import unicodedata

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
from IPython.display import display


# ============================================================
# CONFIGURAÇÃO
# ============================================================

diretorio = Path(
    r"C:\Users\gabri\OneDrive\Cerrado\SBSR\base_cartografica\analise_18_25_pos_vagner"
)

arquivo_grid = diretorio / "inputs" / "grid_1km_cerrado.geojson"
arquivo_focos = diretorio / "inputs" / "focos_01012018_31072026.geojson"
arquivo_deter = diretorio / "inputs" / "deter_cerrado_2018_2026.geojson"
diretorio_outputs = diretorio / "outputs"

# AJUSTE para o arquivo de polígonos e o campo com os nomes das ecorregiões.
# O campo deve conter os nomes da lista codigos_ecorregioes abaixo.
arquivo_ecorregioes = diretorio / "inputs" / "ecorregioes_cerrado.geojson"
campo_nome_ecorregiao = "NAME"

epsg_destino = 5880
anos_prodes = tuple(range(2018, 2026))
focos_por_lote = 50_000
deter_por_lote = 500
grids_ecorregioes_por_lote = 50_000
data_sem_foco = "0000/00/00"
data_sem_deter = "0000/00/00"

codigos_ecorregioes = {
    "Alto Parnaíba": 1,
    "Alto São Francisco": 2,
    "Araguaia Tocantins": 3,
    "Bananal": 4,
    "Basaltos do Paraná": 5,
    "Bico do Papagaio": 6,
    "Centro-norte Piauiense": 7,
    "Chapada dos Parecis": 8,
    "Chapadão do São Francisco": 9,
    "Complexo Bodoquena": 10,
    "Costeiro": 11,
    "Depressão Cárstica do São Francisco": 12,
    "Depressão Cuiabana": 13,
    "Jequitinhonha": 14,
    "Paracatu": 15,
    "Paraná Guimarães": 16,
    "Planalto Central": 17,
    "Vão do Paranã": 18,
    "Floresta de Cocais": 19,
    "Parnaguá": 20,
}

quantidades_ecorregioes = {
    0: "NENHUMA",
    1: "UMA",
    2: "DUAS",
    3: "TRÊS",
    4: "QUATRO",
    5: "CINCO",
    6: "SEIS",
    7: "SETE",
    8: "OITO",
    9: "NOVE",
    10: "DEZ",
}

# True: substitui arquivos de saída existentes.
substituir_outputs = True

# Mantenha None quando o CRS de origem estiver correto no arquivo.
# Altere apenas se souber que a identificação do CRS está errada.
crs_origem_grid = None
crs_origem_focos = None
crs_origem_deter = None
crs_origem_ecorregioes = None


# ============================================================
# TIMER
# ============================================================

def tempo_texto(segundos):
    horas, resto = divmod(int(segundos), 3600)
    minutos, segundos = divmod(resto, 60)
    return f"{horas:02d}:{minutos:02d}:{segundos:02d}"


inicio_total = perf_counter()
registro_tempos = []


@contextmanager
def medir(titulo):
    inicio_etapa = perf_counter()
    print(f"\n{titulo}: iniciado", flush=True)
    estado = "OK"

    try:
        yield
    except BaseException:
        estado = "ERRO"
        raise
    finally:
        duracao = perf_counter() - inicio_etapa

        registro_tempos.append({
            "ETAPA": titulo,
            "STATUS": estado,
            "TEMPO": tempo_texto(duracao),
        })

        print(
            f"{titulo}: {estado}"
            f" | duração {tempo_texto(duracao)}"
            f" | total {tempo_texto(perf_counter() - inicio_total)}",
            flush=True,
        )


# ============================================================
# LEITURA DOS ARQUIVOS
# ============================================================

def carregar(caminho, atributos, crs_informado=None):
    caminho = Path(caminho)

    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    campos = list(pyogrio.read_info(caminho)["fields"])
    nomes = {}

    # Reconhece também campos com espaços extras, como "DataHora ".
    for atributo in atributos:
        correspondentes = [
            campo for campo in campos
            if campo.strip() == atributo
        ]

        if len(correspondentes) != 1:
            raise ValueError(
                f"{caminho.name}: campo {atributo!r} ausente ou ambíguo."
                f"\nCampos disponíveis: {campos}"
            )

        nomes[correspondentes[0]] = atributo

    camada = gpd.read_file(
        caminho,
        engine="pyogrio",
        columns=list(nomes),
    ).rename(columns=nomes)

    if camada.geometry.name != "geometry":
        camada = camada.rename_geometry("geometry")

    camada = camada[
        list(atributos) + ["geometry"]
    ].reset_index(drop=True)

    if crs_informado is not None:
        camada = camada.set_crs(
            crs_informado,
            allow_override=True,
        )

    if camada.crs is None:
        raise ValueError(
            f"{caminho.name}: informe o CRS real de origem na configuração."
        )

    if len(camada) and camada.crs.is_geographic:
        x0, y0, x1, y1 = camada.total_bounds

        if x0 < -180 or x1 > 180 or y0 < -90 or y1 > 90:
            raise ValueError(
                f"{caminho.name}: coordenadas incompatíveis com o CRS "
                "geográfico informado. Confira o CRS de origem."
            )

    camada = camada.to_crs(epsg=epsg_destino)

    print(
        f"  {caminho.name}: {len(camada):,} feições"
        f" | EPSG:{epsg_destino}",
        flush=True,
    )

    return camada


# ============================================================
# VALIDAÇÃO E PREPARAÇÃO
# ============================================================

def conferir_geometrias(camada, nome, tipos):
    problema = (
        camada.geometry.isna()
        | camada.geometry.is_empty
        | ~camada.geometry.is_valid
        | ~camada.geom_type.isin(tipos)
    )

    if problema.any():
        raise ValueError(
            f"{nome}: {int(problema.sum())} geometrias ausentes, vazias, "
            "inválidas ou de tipo incorreto. Corrija o arquivo de entrada."
        )


def preparar_inputs(grade, pontos):
    if grade.empty:
        raise ValueError("O arquivo de grid está vazio.")

    if (
        grade["grid_id"].isna().any()
        or not grade["grid_id"].is_unique
    ):
        raise ValueError("grid_id precisa ser preenchido e único.")

    conferir_geometrias(
        grade, "Grid", {"Polygon", "MultiPolygon"}
    )
    conferir_geometrias(
        pontos, "Focos", {"Point"}
    )

    if (grade.geometry.area <= 0).any():
        raise ValueError("Existem grids sem área.")

    # Descarta a hora, mantendo apenas yyyy-mm-dd.
    texto = (
        pontos["DataHora"]
        .astype("string")
        .str.strip()
        .str[:10]
    )

    formato_valido = texto.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}",
        na=False,
    )

    datas = pd.to_datetime(
        texto.where(formato_valido),
        format="%Y-%m-%d",
        errors="coerce",
    )

    if datas.isna().any():
        exemplos = (
            pontos.loc[datas.isna(), "DataHora"]
            .head(5)
            .tolist()
        )

        raise ValueError(
            f"DataHora: {int(datas.isna().sum())} datas ausentes ou inválidas."
            f"\nExemplos: {exemplos}"
        )

    # Intervalo global: 01/08/2018 até 31/07/2026.
    inicio = pd.Timestamp(f"{anos_prodes[0]}-08-01")
    limite = pd.Timestamp(f"{anos_prodes[-1] + 1}-08-01")

    selecionar = (datas >= inicio) & (datas < limite)

    print(
        f"  Focos no intervalo: {int(selecionar.sum()):,}"
    )
    print(
        f"  Focos fora do intervalo: {int((~selecionar).sum()):,}"
    )

    pontos = pontos.loc[selecionar, ["geometry"]].copy()
    pontos["_data"] = datas.loc[selecionar].to_numpy()
    pontos = pontos.reset_index(drop=True)

    grade = grade[
        ["grid_id", "borda", "geometry"]
    ].reset_index(drop=True)

    return grade, pontos


def preparar_deter(poligonos):
    descartar = (
        poligonos.geometry.isna()
        | poligonos.geometry.is_empty
        | ~poligonos.geometry.is_valid
        | ~poligonos.geom_type.isin({"Polygon", "MultiPolygon"})
    )
    print(f"  DETER descartados por geometria: {int(descartar.sum()):,}")
    poligonos = poligonos.loc[~descartar].copy()

    # Área TOTAL de cada feição em EPSG:5880; não recorta o DETER pelo grid.
    areas = poligonos.geometry.area
    sem_area = (areas <= 0) | ~np.isfinite(areas)
    print(f"  DETER descartados por área: {int(sem_area.sum()):,}")
    poligonos = poligonos.loc[~sem_area].copy()
    areas = areas.loc[~sem_area]
    print(f"  DETER com geometria utilizável: {len(poligonos):,}")

    texto = poligonos["VIEW_DATE"].astype("string").str.strip().str[:10]
    formato_valido = texto.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
    datas = pd.to_datetime(
        texto.where(formato_valido), format="%Y-%m-%d", errors="coerce"
    )
    if datas.isna().any():
        exemplos = poligonos.loc[datas.isna(), "VIEW_DATE"].head(5).tolist()
        raise ValueError(
            f"VIEW_DATE: {int(datas.isna().sum())} datas ausentes ou inválidas."
            f"\nExemplos: {exemplos}"
        )

    inicio = pd.Timestamp(f"{anos_prodes[0]}-08-01")
    limite = pd.Timestamp(f"{anos_prodes[-1] + 1}-08-01")
    selecionar = (datas >= inicio) & (datas < limite)
    print(f"  DETER no intervalo: {int(selecionar.sum()):,}")
    print(f"  DETER fora do intervalo: {int((~selecionar).sum()):,}")

    resultado = poligonos.loc[selecionar, ["geometry"]].copy()
    resultado["_data"] = datas.loc[selecionar].to_numpy()
    resultado["_area_m2"] = areas.loc[selecionar].to_numpy()
    resultado["_ordem"] = poligonos.index[selecionar].to_numpy(dtype=np.int64)
    return resultado.reset_index(drop=True)


# ============================================================
# PREPARAÇÃO E CRUZAMENTO GRID × ECORREGIÕES
# ============================================================

def normalizar_nome_ecorregiao(valor):
    if pd.isna(valor):
        return ""

    # Aceita diferenças de acentuação, maiúsculas e espaços extras.
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )
    for hifen in ("‐", "‑", "–", "—", "−"):
        texto = texto.replace(hifen, "-")
    return " ".join(texto.casefold().split())


def preparar_ecorregioes(poligonos):
    if poligonos.empty:
        raise ValueError("O arquivo de ecorregiões está vazio.")

    conferir_geometrias(
        poligonos, "Ecorregiões", {"Polygon", "MultiPolygon"}
    )
    areas = poligonos.geometry.area
    if ((areas <= 0) | ~np.isfinite(areas)).any():
        raise ValueError("Existem ecorregiões sem área válida.")

    mapa_codigos = {
        normalizar_nome_ecorregiao(nome): codigo
        for nome, codigo in codigos_ecorregioes.items()
    }
    nomes = poligonos[campo_nome_ecorregiao].map(
        normalizar_nome_ecorregiao
    )
    codigos = nomes.map(mapa_codigos)

    if codigos.isna().any():
        exemplos = (
            poligonos.loc[codigos.isna(), campo_nome_ecorregiao]
            .drop_duplicates().head(10).tolist()
        )
        raise ValueError(
            f"Ecorregiões: {int(codigos.isna().sum())} feições com nome "
            "ausente ou não incluído em codigos_ecorregioes."
            f"\nExemplos: {exemplos}"
        )

    resultado = poligonos[["geometry"]].copy()
    resultado["_eco_codigo"] = codigos.to_numpy(dtype=np.int64)
    return resultado.reset_index(drop=True)


def resumir_ecorregioes(pares):
    # Conta e lista códigos distintos, mesmo com vários polígonos da mesma eco.
    codigos_por_grid = (
        pares.drop_duplicates(["_grid_pos", "_eco_codigo"])
        .sort_values(["_grid_pos", "_eco_codigo"])
        .groupby("_grid_pos", sort=False)["_eco_codigo"]
        .agg(list)
    )
    resumo = pd.DataFrame(index=codigos_por_grid.index)
    resumo["ECO_UNICA"] = codigos_por_grid.map(len).map(
        quantidades_ecorregioes
    )
    resumo["ECORREGIOES"] = codigos_por_grid.map(
        lambda codigos: "-".join(str(codigo) for codigo in codigos)
    )
    return resumo


def adicionar_ecorregioes(grade, poligonos, lote):
    if not isinstance(lote, int) or lote < 1:
        raise ValueError(
            "grids_ecorregioes_por_lote deve ser um inteiro positivo."
        )

    # Grids sem sobreposição permanecem na saída com estes valores.
    grade["ECO_UNICA"] = quantidades_ecorregioes[0]
    grade["ECORREGIOES"] = ""
    ecos_para_consulta = poligonos[["_eco_codigo", "geometry"]].copy()
    _ = ecos_para_consulta.sindex
    ultimo_aviso = perf_counter()

    for inicio_lote in range(0, len(grade), lote):
        trecho = grade.iloc[inicio_lote:inicio_lote + lote][["geometry"]]
        cruzamento = gpd.sjoin(
            trecho,
            ecos_para_consulta,
            how="inner",
            predicate="intersects",
        )

        if not cruzamento.empty:
            posicoes_ecos = cruzamento["index_right"].to_numpy(
                dtype=np.int64
            )
            geometrias_ecos = ecos_para_consulta.geometry.iloc[posicoes_ecos]

            # Uma eco precisa ocupar área dentro do grid.
            # Toques apenas em linhas ou pontos de borda não contam.
            # align=False compara cada par na ordem da junção espacial.
            areas_intersecao = cruzamento.geometry.intersection(
                geometrias_ecos, align=False
            ).area.to_numpy()

            pares = pd.DataFrame({
                "_grid_pos": cruzamento.index.to_numpy(dtype=np.int64),
                "_eco_codigo": cruzamento["_eco_codigo"].to_numpy(
                    dtype=np.int64
                ),
            })
            resumo = resumir_ecorregioes(
                pares.loc[areas_intersecao > 0]
            )
            grade.loc[
                resumo.index, ["ECO_UNICA", "ECORREGIOES"]
            ] = resumo
            del resumo, pares, areas_intersecao, geometrias_ecos

        fim = min(inicio_lote + lote, len(grade))
        agora = perf_counter()
        if agora - ultimo_aviso >= 10 or fim == len(grade):
            print(
                f"  Grids consultados para ecorregiões: {fim:,}/{len(grade):,}",
                flush=True,
            )
            ultimo_aviso = agora
        del cruzamento

    print(
        "  Grids sem ecorregião: "
        f"{int((grade['ECO_UNICA'] == quantidades_ecorregioes[0]).sum()):,}",
        flush=True,
    )
    return grade


# ============================================================
# CRUZAMENTO GRID × FOCOS
# ============================================================

def resultado_anual(grade, grade_consulta, pontos, ano, lote):
    if not isinstance(lote, int) or lote < 1:
        raise ValueError("focos_por_lote deve ser um inteiro positivo.")

    inicio = pd.Timestamp(f"{ano}-08-01")
    limite = pd.Timestamp(f"{ano + 1}-08-01")

    anuais = pontos.loc[
        (pontos["_data"] >= inicio)
        & (pontos["_data"] < limite)
    ]

    total_grids = len(grade)

    quantidade = np.zeros(total_grids, dtype=np.int64)

    primeiro_dia = np.full(
        total_grids,
        np.iinfo(np.int64).max,
        dtype=np.int64,
    )

    ultimo_dia = np.full(
        total_grids,
        np.iinfo(np.int64).min,
        dtype=np.int64,
    )

    focos_localizados = 0
    ultimo_aviso = perf_counter()

    for inicio_lote in range(0, len(anuais), lote):
        trecho = anuais.iloc[
            inicio_lote:inicio_lote + lote
        ]

        # Conta focos dentro do grid ou sobre sua borda.
        # Um foco na divisa pode contar em mais de um grid.
        cruzamento = gpd.sjoin(
            trecho,
            grade_consulta,
            how="inner",
            predicate="intersects",
        )

        if not cruzamento.empty:
            focos_localizados += cruzamento.index.nunique()

            resumo = cruzamento.groupby("index_right")["_data"].agg(
                quantidade="size",
                primeira="min",
                ultima="max",
            )

            posicoes = resumo.index.to_numpy(dtype=np.int64)

            quantidade[posicoes] += resumo[
                "quantidade"
            ].to_numpy(dtype=np.int64)

            primeiras = (
                resumo["primeira"]
                .to_numpy(dtype="datetime64[D]")
                .astype(np.int64)
            )

            ultimas = (
                resumo["ultima"]
                .to_numpy(dtype="datetime64[D]")
                .astype(np.int64)
            )

            primeiro_dia[posicoes] = np.minimum(
                primeiro_dia[posicoes],
                primeiras,
            )

            ultimo_dia[posicoes] = np.maximum(
                ultimo_dia[posicoes],
                ultimas,
            )

        fim = min(inicio_lote + lote, len(anuais))
        agora = perf_counter()

        if agora - ultimo_aviso >= 10 or fim == len(anuais):
            print(
                f"  Focos consultados: {fim:,}/{len(anuais):,}",
                flush=True,
            )
            ultimo_aviso = agora

        del cruzamento

    presentes = quantidade > 0

    primeira_data = np.full(
        total_grids,
        data_sem_foco,
        dtype="U10",
    )

    ultima_data = np.full(
        total_grids,
        data_sem_foco,
        dtype="U10",
    )

    if presentes.any():
        for dias, destino in [
            (primeiro_dia, primeira_data),
            (ultimo_dia, ultima_data),
        ]:
            datas_formatadas = np.datetime_as_string(
                dias[presentes].astype("datetime64[D]"),
                unit="D",
            )

            destino[presentes] = np.char.replace(
                datas_formatadas,
                "-",
                "/",
            )

    # Parte sempre de toda a grade, preservando grids sem focos.
    saida = grade.copy()

    saida["FOCO"] = np.where(presentes, "SIM", "NAO")
    saida["QNTD_FOCOS"] = quantidade
    saida["1a_DATA_FOCO"] = primeira_data
    saida["2a_DATA_FOCO"] = ultima_data

    saida = saida[[
        "grid_id",
        "borda",
        "FOCO",
        "QNTD_FOCOS",
        "1a_DATA_FOCO",
        "2a_DATA_FOCO",
        "ECO_UNICA",
        "ECORREGIOES",
        "geometry",
    ]]

    if (
        len(saida) != total_grids
        or not saida["grid_id"].equals(grade["grid_id"])
    ):
        raise RuntimeError(
            "A saída não preservou a quantidade ou a ordem dos grids."
        )

    estatisticas = {
        "ANO_PRODES": f"{ano % 100:02d}-{(ano + 1) % 100:02d}",
        "GRIDS": total_grids,
        "FOCOS_NO_PERIODO": len(anuais),
        "FOCOS_EM_GRID": int(focos_localizados),
        "GRIDS_COM_FOCO": int(presentes.sum()),
        "ATRIBUICOES_FOCOS": int(quantidade.sum()),
    }

    return saida, estatisticas


# ============================================================
# CRUZAMENTO GRID × DETER
# ============================================================

def adicionar_deter(saida, grade_consulta, poligonos, ano, lote):
    if not isinstance(lote, int) or lote < 1:
        raise ValueError("deter_por_lote deve ser um inteiro positivo.")

    inicio = pd.Timestamp(f"{ano}-08-01")
    limite = pd.Timestamp(f"{ano + 1}-08-01")
    anuais = poligonos.loc[
        (poligonos["_data"] >= inicio) & (poligonos["_data"] < limite)
    ]
    total_grids = len(saida)
    maior_area = np.full(total_grids, -1.0, dtype=np.float64)
    melhor_dia = np.full(total_grids, np.iinfo(np.int64).max, dtype=np.int64)
    melhor_ordem = np.full(total_grids, np.iinfo(np.int64).max, dtype=np.int64)
    poligonos_localizados = 0
    ultimo_aviso = perf_counter()

    for inicio_lote in range(0, len(anuais), lote):
        trecho = anuais.iloc[inicio_lote:inicio_lote + lote]

        # Qualquer interseção conta, inclusive toque na borda.
        # Um mesmo polígono pode participar de vários grids.
        # A junção não recorta nem divide o polígono DETER.
        cruzamento = gpd.sjoin(
            trecho, grade_consulta, how="inner", predicate="intersects"
        )
        if not cruzamento.empty:
            poligonos_localizados += cruzamento.index.nunique()

            # Candidato de maior área TOTAL para cada grid neste lote.
            # Empate: data mais antiga; depois, primeira feição no input.
            candidatos = cruzamento.sort_values(
                ["index_right", "_area_m2", "_data", "_ordem"],
                ascending=[True, False, True, True],
            ).drop_duplicates("index_right", keep="first")

            posicoes = candidatos["index_right"].to_numpy(dtype=np.int64)
            areas = candidatos["_area_m2"].to_numpy(dtype=np.float64)
            dias = candidatos["_data"].to_numpy(dtype="datetime64[D]").astype(np.int64)
            ordens = candidatos["_ordem"].to_numpy(dtype=np.int64)

            # Compara com os melhores candidatos dos lotes anteriores.
            empate_area = areas == maior_area[posicoes]
            substituir = (
                (areas > maior_area[posicoes])
                | (empate_area & (dias < melhor_dia[posicoes]))
                | (empate_area & (dias == melhor_dia[posicoes])
                   & (ordens < melhor_ordem[posicoes]))
            )
            atualizar = posicoes[substituir]
            maior_area[atualizar] = areas[substituir]
            melhor_dia[atualizar] = dias[substituir]
            melhor_ordem[atualizar] = ordens[substituir]
            del candidatos

        fim = min(inicio_lote + lote, len(anuais))
        agora = perf_counter()
        if agora - ultimo_aviso >= 10 or fim == len(anuais):
            print(f"  DETER consultados: {fim:,}/{len(anuais):,}", flush=True)
            ultimo_aviso = agora
        del cruzamento

    presentes = maior_area >= 0
    datas_saida = np.full(total_grids, data_sem_deter, dtype="U10")
    if presentes.any():
        datas_formatadas = np.datetime_as_string(
            melhor_dia[presentes].astype("datetime64[D]"), unit="D"
        )
        datas_saida[presentes] = np.char.replace(datas_formatadas, "-", "/")

    saida["DETER"] = np.where(presentes, "SIM", "NAO")
    # Presença por célula neste ano PRODES: basta um polígono DETER.
    # A célula recebe um único SIM, mesmo com vários polígonos.
    saida["presenca_deter"] = np.where(presentes, "SIM", "NAO")
    saida["DATA_DETER"] = datas_saida
    estatisticas_deter = {
        "DETER_NO_PERIODO": len(anuais),
        "DETER_EM_GRID": int(poligonos_localizados),
        "GRIDS_COM_DETER": int(presentes.sum()),
    }
    return saida, estatisticas_deter


# ============================================================
# EXPORTAÇÃO
# ============================================================

def gravar_geojson(saida, destino):
    destino = Path(destino)

    if destino.exists() and not substituir_outputs:
        raise FileExistsError(f"Saída já existe: {destino}")

    destino.parent.mkdir(parents=True, exist_ok=True)

    temporario = destino.with_name(
        destino.stem + ".temporario.geojson"
    )

    try:
        # Preserva EPSG:5880 no GeoJSON e não exporta o índice.
        saida.to_file(
            temporario,
            driver="GeoJSON",
            engine="pyogrio",
            index=False,
            layer_options={"RFC7946": "NO"},
        )

        os.replace(temporario, destino)

    finally:
        if temporario.exists():
            temporario.unlink()

    print(
        f"  Gravado: {destino.name} | {len(saida):,} grids",
        flush=True,
    )


# ============================================================
# EXECUÇÃO
# ============================================================

inicio_total = perf_counter()
registro_tempos.clear()

with medir("Etapa 1 — leitura e reprojeção"):
    grid = carregar(
        arquivo_grid,
        ["grid_id", "borda"],
        crs_origem_grid,
    )

    focos = carregar(
        arquivo_focos,
        ["DataHora"],
        crs_origem_focos,
    )

    deter = carregar(
        arquivo_deter,
        ["VIEW_DATE"],
        crs_origem_deter,
    )

    ecorregioes = carregar(
        arquivo_ecorregioes,
        [campo_nome_ecorregiao],
        crs_origem_ecorregioes,
    )

with medir("Etapa 2 — validação e preparação das datas"):
    grid, focos = preparar_inputs(grid, focos)
    deter = preparar_deter(deter)
    ecorregioes = preparar_ecorregioes(ecorregioes)

# A classificação é espacial e independe do ano: calcula somente uma vez.
with medir("Etapa 3 — grid × ecorregiões"):
    grid = adicionar_ecorregioes(
        grid, ecorregioes, lote=grids_ecorregioes_por_lote
    )
    del ecorregioes
    gc.collect()

with medir("Etapa 4 — índice espacial dos grids"):
    grids_para_consulta = grid[["geometry"]].copy()
    _ = grids_para_consulta.sindex

resultados = []

for ano in anos_prodes:
    identificacao = f"{ano % 100:02d}-{(ano + 1) % 100:02d}"
    inicio_ano = perf_counter()

    with medir(f"Ano PRODES {identificacao} — grid × focos"):
        saida, estatisticas = resultado_anual(
            grid,
            grids_para_consulta,
            focos,
            ano,
            lote=focos_por_lote,
        )

    with medir(f"Ano PRODES {identificacao} — grid × DETER"):
        saida, estatisticas_deter = adicionar_deter(
            saida,
            grids_para_consulta,
            deter,
            ano,
            lote=deter_por_lote,
        )
        estatisticas.update(estatisticas_deter)

    # Exporta somente os atributos pedidos e a geometria original do grid.
    saida = saida[[
        "grid_id", "borda", "FOCO", "QNTD_FOCOS", "1a_DATA_FOCO",
        "2a_DATA_FOCO", "DETER", "presenca_deter", "DATA_DETER",
        "ECO_UNICA", "ECORREGIOES", "geometry",
    ]]

    caminho_saida = diretorio_outputs / (
        f"ano_prodes_{ano % 100:02d}_{(ano + 1) % 100:02d}.geojson"
    )

    with medir(f"Ano PRODES {identificacao} — gravação"):
        gravar_geojson(saida, caminho_saida)

    estatisticas["TEMPO_ANO"] = tempo_texto(
        perf_counter() - inicio_ano
    )
    estatisticas["ARQUIVO"] = caminho_saida.name

    resultados.append(estatisticas)

    del saida
    gc.collect()

print(
    "\nPROCESSAMENTO CONCLUÍDO"
    f" — tempo total: {tempo_texto(perf_counter() - inicio_total)}"
)

display(pd.DataFrame(resultados))
display(pd.DataFrame(registro_tempos))



Etapa 1 — leitura e reprojeção: iniciado
  grid_1km_cerrado.geojson: 2,024,694 feições | EPSG:5880
  focos_01012018_31072026.geojson: 478,391 feições | EPSG:5880
  deter_cerrado_2018_2026.geojson: 133,687 feições | EPSG:5880
  ecorregioes_cerrado.geojson: 20 feições | EPSG:5880
Etapa 1 — leitura e reprojeção: OK | duração 00:01:10 | total 00:01:10

Etapa 2 — validação e preparação das datas: iniciado
  Focos no intervalo: 466,268
  Focos fora do intervalo: 12,123
  DETER descartados por geometria: 123
  DETER descartados por área: 0
  DETER com geometria utilizável: 133,564
  DETER no intervalo: 119,057
  DETER fora do intervalo: 14,507
Etapa 2 — validação e preparação das datas: OK | duração 00:00:04 | total 00:01:15

Etapa 3 — grid × ecorregiões: iniciado
  Grids consultados para ecorregiões: 50,000/2,024,694
  Grids consultados para ecorregiões: 100,000/2,024,694
  Grids consultados para ecorregiões: 150,000/2,024,694
  Grids consultados para ecorregiões: 200,000/2,024,694
  Grids 

,ANO_PRODES,GRIDS,FOCOS_NO_PERIODO,FOCOS_EM_GRID,GRIDS_COM_FOCO,ATRIBUICOES_FOCOS,DETER_NO_PERIODO,DETER_EM_GRID,GRIDS_COM_DETER,TEMPO_ANO,ARQUIVO
0,18-19,2024694,42166,42166,38713,42167,17275,17275,37409,00:01:14,ano_prodes_18_19.geojson
1,19-20,2024694,62588,62588,57188,62589,11999,11999,25230,00:01:12,ano_prodes_19_20.geojson
2,20-21,2024694,64851,64851,59177,64852,10472,10472,25630,00:01:24,ano_prodes_20_21.geojson
3,21-22,2024694,59031,59031,53519,59031,8502,8502,23451,00:01:18,ano_prodes_21_22.geojson
4,22-23,2024694,55806,55806,50292,55806,11766,11766,28032,00:01:30,ano_prodes_22_23.geojson
5,23-24,2024694,54509,54509,48875,54509,19044,19044,36920,00:01:29,ano_prodes_23_24.geojson
6,24-25,2024694,74529,74529,67218,74529,15238,15238,27715,00:01:23,ano_prodes_24_25.geojson
7,25-26,2024694,52788,52788,48001,52788,24761,24761,36785,00:01:22,ano_prodes_25_26.geojson


,ETAPA,STATUS,TEMPO
0,Etapa 1 — leitura e reprojeção,OK,00:01:10
1,Etapa 2 — validação e preparação das datas,OK,00:00:04
2,Etapa 3 — grid × ecorregiões,OK,01:24:27
3,Etapa 4 — índice espacial dos grids,OK,00:00:00
4,Ano PRODES 18-19 — grid × focos,OK,00:00:01
5,Ano PRODES 18-19 — grid × DETER,OK,00:00:01
6,Ano PRODES 18-19 — gravação,OK,00:01:12
7,Ano PRODES 19-20 — grid × focos,OK,00:00:01
8,Ano PRODES 19-20 — grid × DETER,OK,00:00:01
9,Ano PRODES 19-20 — gravação,OK,00:01:10
